# 🌍 AI Live Text Translator

A multilingual text translation system using the **NLLB-200 Transformer model**.

### Current features
- GPU acceleration with CUDA
- Multilingual translation
- User-selected source and target languages
- Bidirectional translation
- Basic automatic language detection
- Clean, reusable translation function

### Current model
`facebook/nllb-200-distilled-600M`


## 1. Install Dependencies


In [ ]:
!pip install -q transformers sentencepiece langdetect


## 2. Import Libraries


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from langdetect import detect


## 3. Configure Device

We use the NVIDIA GPU when available.


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 4. Model Configuration


In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"


## 5. Load NLLB Model

NLLB is a multilingual sequence-to-sequence Transformer designed for machine translation.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Model device:", next(model.parameters()).device)


## 6. Supported Languages

NLLB uses its own language codes. We keep the mapping in one place so the translation function and future UI can reuse it.


In [ ]:
LANGUAGE_MAP = {
    "English": "eng_Latn",
    "German": "deu_Latn",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "Italian": "ita_Latn",
    "Hindi": "hin_Deva",
}

print("Supported languages:")
for language in LANGUAGE_MAP:
    print("-", language)


## 7. Translation Function

The function accepts **both source and target languages**, so the translator is not limited to English → German.


In [ ]:
def translate_text(text, source_language, target_language):
    # Translate text from source_language to target_language.

    if not text or not text.strip():
        return ""

    if source_language not in LANGUAGE_MAP:
        raise ValueError(f"Unsupported source language: {source_language}")

    if target_language not in LANGUAGE_MAP:
        raise ValueError(f"Unsupported target language: {target_language}")

    if source_language == target_language:
        return text

    source_code = LANGUAGE_MAP[source_language]
    target_code = LANGUAGE_MAP[target_language]

    # Tell NLLB which language the input uses
    tokenizer.src_lang = source_code

    # Convert text to token IDs
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Move tensors to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate translation without calculating gradients
    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(
                target_code
            )
        )

    # Convert generated tokens back to text
    translation = tokenizer.batch_decode(
        translated_tokens,
        skip_special_tokens=True
    )[0]

    return translation


## 8. Basic Translation Test


In [ ]:
text = "I am learning artificial intelligence."

result = translate_text(
    text,
    "English",
    "German"
)

print("Source:", text)
print("Translation:", result)


## 9. Bidirectional and Multilingual Tests

These tests verify that source and target languages can be selected independently.


In [ ]:
test_cases = [
    ("I am learning artificial intelligence.", "English", "German"),
    ("Ich lerne künstliche Intelligenz.", "German", "English"),
    ("I love programming.", "English", "French"),
    ("J'aime programmer.", "French", "English"),
    ("Ich studiere Informatik.", "German", "Spanish"),
    ("Estoy estudiando informática.", "Spanish", "German"),
]

for text, source, target in test_cases:
    result = translate_text(text, source, target)

    print(f"{source} → {target}")
    print("Input :", text)
    print("Output:", result)
    print("-" * 60)


## 10. Automatic Language Detection

For the prototype, `langdetect` provides the detected language code. Short or ambiguous text can sometimes be misclassified.


In [ ]:
LANGUAGE_DETECT_MAP = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "es": "Spanish",
    "it": "Italian",
    "hi": "Hindi",
}


def detect_language(text):
    if not text or not text.strip():
        return None

    detected_code = detect(text)

    return LANGUAGE_DETECT_MAP.get(detected_code)


## 11. Test Language Detection


In [ ]:
test_texts = [
    "I am learning artificial intelligence.",
    "Ich lerne künstliche Intelligenz.",
    "Bonjour, comment allez-vous ?",
    "Estoy aprendiendo inteligencia artificial.",
]

for text in test_texts:
    detected = detect_language(text)

    print("Text:", text)
    print("Detected:", detected)
    print("-" * 60)


## 12. Auto-Detection + Translation Helper


In [ ]:
def auto_translate(text, target_language):
    # Detect source language automatically and translate.

    source_language = detect_language(text)

    if source_language is None:
        raise ValueError(
            "The input language is not currently supported."
        )

    translation = translate_text(
        text,
        source_language,
        target_language
    )

    return source_language, translation


## 13. Test Automatic Translation


In [ ]:
text = "Ich lerne künstliche Intelligenz."

source, result = auto_translate(
    text,
    "English"
)

print("Detected source:", source)
print("Translation:", result)


## 14. Project Architecture

```text
User Text
   │
   ├── Manual Mode
   │      ├── Source Language
   │      └── Target Language
   │
   └── Auto Mode
          │
          └── Language Detection
                    │
                    ▼
              NLLB Transformer
                    │
                    ▼
             Translated Text
```

### Next development stages
1. Streamlit/Gradio interface
2. Source/target dropdowns
3. Swap languages button
4. Live translation
5. Translation history
6. BLEU/chrF evaluation
7. Model comparison
8. Deployment
